# 摘要记忆（Summary Memory）

> **用 LLM 生成的滚动摘要压缩较早的轮次：以逐字复述为代价，换来不限长度的对话。**

在[上一个 notebook](../02_sliding_window_memory/sliding_window_memory.ipynb)里，我们看到**滑动窗口记忆（Sliding Window Memory）**靠直接丢弃旧消息来控制成本。这种硬截断意味着智能体甚至*不知道*自己忘了东西。

**摘要记忆**换了个思路。它不丢弃历史，而是*压缩*历史。想象你在读一本很长的书，每读完一章就在结尾写一页摘要。你没法再逐字引用原书，但仍然知道发生了什么。系统周期性地用一次额外的 LLM 调用，把较早的消息浓缩成一段不断更新的文字摘要。智能体失去了准确措辞，却能在任意长的对话中保住要点（关键事实、决策和上下文）。

**代价是：**摘要是有损的（无法完美还原原文）。每轮压缩都可能丢细节、转移重点，或悄悄扭曲事实。多轮循环之后，**摘要漂移（summary drift）**会不断累积，智能体的「记忆」可能偏离实际发生的事。

**读完本 notebook，你将理解：**
- 如何用 Anthropic SDK 从零构建一个滚动摘要记忆系统。
- 摘要循环：何时触发、用什么提示词、摘要如何演化。
- 摘要在长对话中如何漂移，附一个受控实验和可视化。
- 缓解漂移和信息丢失的实用方法。


## 关键概念

- **滚动摘要（rolling summary）**：一个随对话增长而增量更新的文本块。每次更新把新消息并入现有摘要。
- **摘要提示词（summarizer prompt）**：让 LLM 生成摘要的指令。它的措辞决定保留什么（事实、决策、语气）、丢弃什么。
- **刷新触发器（refresh trigger）**：决定*何时*重新摘要的规则。常见选项：每 *n* 条消息后触发、缓冲区超过 token 阈值时触发，或每轮触发。
- **摘要漂移（summary drift）**：反复摘要循环导致的逐渐失真。随时间推移，细节被淡化、合并，甚至彻底丢失。
- **压缩比（compression ratio）**：摘要比它替代的原始消息短了多少。压缩比越高，信息损失越大。
- **缓冲区（buffer zone）**：与摘要并存的、逐字保留的近期消息。它让 LLM 对最近的交流拥有精确上下文，而更早的上下文则存放在压缩摘要里。


## 架构

<p align="center">
  <img src="../../images/diagrams/03_summary_memory.svg" alt="摘要记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
sequenceDiagram
    participant U as User
    participant B as Message Buffer
    participant S as Summary Store
    participant LLM as Claude (Chat)
    participant SLLM as Claude (Summarizer)

    Note over S: Summary: "" (empty)
    Note over B: Buffer: []

    U->>B: msg 1
    B->>LLM: [summary="", msg1]
    LLM-->>B: msg 2
    Note over B: Buffer: [msg1, msg2]

    U->>B: msg 3
    B->>LLM: [summary="", msg1..msg3]
    LLM-->>B: msg 4
    Note over B: Buffer: [msg1..msg4] - trigger!

    rect rgb(255, 245, 230)
        Note over B,SLLM: 🔄 Summarization Cycle
        B->>SLLM: "Summarize: {old_summary} + {msg1..msg4}"
        SLLM-->>S: Updated summary
        B->>B: Clear buffer
    end

    U->>B: msg 5
    B->>LLM: [summary="...", msg5]
    LLM-->>B: msg 6
    Note over B: Buffer: [msg5, msg6]
    Note over S: Summary carries forward<br/>compressed history
```

</details>

In [ ]:
# 安装所需包（只需运行一次）
%pip install -q python-dotenv matplotlib pandas numpy deepagents langchain-openai

## 环境设置

导入 Anthropic SDK，从 `.env` 文件加载你的 API key。


In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)



def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )



def usage_of(reply, prompt_messages: list[dict]) -> dict:
    """读取一轮对话的 token 用量；端点未返回时用 tiktoken 估算。"""
    import tiktoken

    usage = getattr(reply, "usage_metadata", None) or {}
    encoder = tiktoken.get_encoding("cl100k_base")
    prompt_text = "".join(str(m.get("content", "")) for m in prompt_messages)
    reply_text = to_text(reply.content)
    return {
        "input_tokens": int(usage.get("input_tokens") or len(encoder.encode(prompt_text))),
        "output_tokens": int(usage.get("output_tokens") or len(encoder.encode(reply_text))),
    }


print("✓ Model ready")


## 核心实现

设计分两部分：

1. **缓冲区**：逐字保存的近期消息（一个 dict 列表）。
2. **摘要**：一个字符串，压缩了所有更早的消息。

当缓冲区达到 `max_buffer_size` 条消息时，我们用*摘要提示词*调用 LLM。该提示词把当前摘要和缓冲区融合成一份新摘要，然后清空缓冲区。

每轮对话时，LLM 收到：`[system: summary_context] + buffer_messages`。


In [ ]:
SUMMARIZER_PROMPT = """You are a conversation summarizer. Given the existing summary and new messages,
produce an updated summary that:
1. Preserves all key FACTS (names, numbers, preferences, decisions).
2. Notes any open questions or unresolved topics.
3. Stays concise (aim for 2-5 sentences).

Current summary:
{summary}

New messages:
{messages}

Write only the updated summary, nothing else."""




现在定义 `SummaryMemory` 类。构造函数建立两个存储区：`summary` 字符串（初始为空）和保存近期消息的 `buffer` 列表。`_summarize` 方法调用 LLM，把当前摘要加缓冲区压缩成一份更短的新摘要。


In [ ]:
class SummaryMemory:
    """通过 LLM 压缩较早轮次的滚动摘要记忆。"""

    def __init__(
        self,
        model,
        max_buffer_size: int = 6,
        system_prompt: str | None = None,
    ):
        self.model = model
        self.system_prompt = system_prompt
        self.max_buffer_size = max_buffer_size

        # 状态
        self.summary: str = ""
        self.buffer: list[dict] = []

        # 追踪
        self.full_history: list[dict] = []
        self.summary_history: list[str] = []  # 每个摘要版本
        self.turn_token_usage: list[dict] = []
        self._summary_calls = 0

    # ── 摘要 ─────────────────────────────────────────────────────
    def _format_messages_for_summary(self, messages: list[dict]) -> str:
        lines = []
        for msg in messages:
            role = "User" if msg["role"] == "user" else "Assistant"
            lines.append(f"{role}: {msg['content']}")
        return "\n".join(lines)

    def _summarize(self) -> str:
        """把当前摘要和缓冲区压缩成一份新摘要。"""
        messages_text = self._format_messages_for_summary(self.buffer)
        prompt = SUMMARIZER_PROMPT.format(
            summary=self.summary or "(no prior summary)",
            messages=messages_text,
        )
        response = self.model.invoke([{"role": "user", "content": prompt}])
        self._summary_calls += 1
        return to_text(response.content).strip()


`chat` 方法是核心所在。它构建一个包含滚动摘要的系统提示词，把缓冲区发给 LLM，然后检查缓冲区是否超过 `max_buffer_size`。如果超过，就触发一次摘要循环并清空缓冲区。


In [ ]:
# ── 对话 ──────────────────────────────────────────────────────
def _chat(self, user_input: str) -> str:
    user_msg = {"role": "user", "content": user_input}
    self.buffer.append(user_msg)
    self.full_history.append(user_msg)

    # 构建带摘要上下文的系统提示词
    system_parts = []
    if self.system_prompt:
        system_parts.append(self.system_prompt)
    if self.summary:
        system_parts.append(f"Conversation summary so far:\n{self.summary}")
    system = "\n\n".join(system_parts) or "You are a helpful assistant."

    # 摘要随对话更新，系统提示词每轮都变，所以每轮重建智能体
    agent = create_deep_agent(model=self.model, system_prompt=system)
    result = agent.invoke({"messages": list(self.buffer)})
    reply = result["messages"][-1]
    assistant_text = to_text(reply.content)

    assistant_msg = {"role": "assistant", "content": assistant_text}
    self.buffer.append(assistant_msg)
    self.full_history.append(assistant_msg)

    usage = usage_of(reply, list(self.buffer))
    self.turn_token_usage.append({
        "turn": len(self.turn_token_usage) + 1,
        "input_tokens": usage["input_tokens"],
        "output_tokens": usage["output_tokens"],
        "buffer_msgs": len(self.buffer),
        "summary_len": len(self.summary),
    })

    # 检查是否需要摘要
    if len(self.buffer) >= self.max_buffer_size:
        self.summary = self._summarize()
        self.summary_history.append(self.summary)
        self.buffer.clear()

    return assistant_text


SummaryMemory.chat = _chat

print("✓ SummaryMemory.chat attached")


最后加上检查和工具方法。`get_context_snapshot` 会精确展示 LLM 在下一次调用时会看到什么：摘要文本加当前缓冲区内容。


In [ ]:
# ── 检查 ──────────────────────────────────────────────────────
def _get_context_snapshot(self) -> dict:
    """返回 LLM 当前看到的内容。"""
    return {
        "summary": self.summary,
        "buffer": list(self.buffer),
        "buffer_size": len(self.buffer),
        "total_messages": len(self.full_history),
        "summary_versions": len(self.summary_history),
    }


def _clear(self) -> None:
    self.summary = ""
    self.buffer.clear()
    self.full_history.clear()
    self.summary_history.clear()
    self.turn_token_usage.clear()
    self._summary_calls = 0


def _repr(self) -> str:
    return (
        f"SummaryMemory(model=llm, buffer={len(self.buffer)} msgs, "
        f"summary={'yes' if self.summary else 'empty'}, "
        f"turns={len(self.full_history) // 2})"
    )


SummaryMemory.get_context_snapshot = _get_context_snapshot
SummaryMemory.clear = _clear
SummaryMemory.__repr__ = _repr

print("✓ SummaryMemory helper methods attached")


## 使用示例：观察摘要的演化

我们用一个较小的缓冲区（`max_buffer_size=4`，即每 2 轮触发一次摘要）。先植入几条事实，再观察它们如何被压缩进摘要。


In [ ]:
mem = SummaryMemory(
    model=llm,
    max_buffer_size=4,  # 每 2 轮摘要一次
    system_prompt="You are a concise assistant. Reply in 1-2 sentences.",
)

conversation = [
    "My name is Carlos and I'm from Buenos Aires.",
    "I work as a marine biologist studying coral reefs.",
    "My favorite programming language is Rust.",
    "What do you know about me so far?",
]

for msg in conversation:
    print(f"👤 User:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    snap = mem.get_context_snapshot()
    print(f"   📊 Buffer: {snap['buffer_size']}/{mem.max_buffer_size} | Summary versions: {snap['summary_versions']}")
    if snap["summary"]:
        print(f"   📝 Current summary: {snap['summary'][:120]}...")
    print()

我们追踪一下摘要随时间的演化。每个版本都是摘要器并入一批消息后的产物。同时打印当前缓冲区的内容。


In [ ]:
print("=== Summary Evolution ===\n")
for i, s in enumerate(mem.summary_history):
    print(f"Version {i+1}:")
    print(f"  {s}")
    print()

print(f"=== Current Buffer ({len(mem.buffer)} messages) ===")
for msg in mem.buffer:
    role = "USER" if msg["role"] == "user" else "ASST"
    print(f"  {role}: {msg['content'][:80]}")

## 实验：长对话中的摘要漂移

摘要漂移是这项技术最重要的失败模式。每轮摘要都有损，误差会累积。第 1 轮明确陈述的事实，可能在摘要 v2 里被弱化，在 v3 里被含糊带过，到 v5 彻底消失。

我们来做一个受控实验：
1. 在前 5 轮植入 **5 条具体事实**。
2. 再继续 **15 轮无关的填充对话**（触发多次重新摘要）。
3. 每次摘要循环后，检查每条原始事实是否仍出现在摘要中。
4. 可视化事实保留率如何随摘要版本退化。


In [ ]:
import re

FACTS = [
    ("My full name is Elena Vasquez.", "elena vasquez"),
    ("I was born on March 15, 1992.", "1992"),
    ("I have exactly three cats named Mochi, Tofu, and Tempeh.", "tempeh"),
    ("My salary is $145,000 per year.", "145"),
    ("I'm allergic to strawberries.", "strawberr"),
]

FILLER_MESSAGES = [
    "What's a good recipe for pasta carbonara?",
    "Tell me about the history of jazz music.",
    "How do black holes form?",
    "What are the main differences between Python and JavaScript?",
    "Can you explain quantum entanglement simply?",
    "What's the tallest building in the world?",
    "How does a combustion engine work?",
    "What are some tips for better sleep?",
    "Tell me about the migration patterns of monarch butterflies.",
    "What causes the northern lights?",
    "How do vaccines work?",
    "What's the deepest point in the ocean?",
    "Tell me about the history of chess.",
    "How does GPS work?",
    "What are prime numbers used for in cryptography?",
]

# 用小缓冲区运行实验，强制产生多次摘要循环
mem_drift = SummaryMemory(
    model=llm,
    max_buffer_size=4,  # 每 2 轮摘要一次
    system_prompt="You are a helpful assistant. Reply concisely in 1-2 sentences.",
)

# 阶段 1：植入事实
print("Phase 1: Planting facts...")
for fact_text, _ in FACTS:
    mem_drift.chat(fact_text)
print(f"  Planted {len(FACTS)} facts. Summary versions so far: {len(mem_drift.summary_history)}")

# 阶段 2：填充对话，触发反复重新摘要
print("\nPhase 2: Filler turns (causing re-summarization)...")
for i, filler in enumerate(FILLER_MESSAGES):
    mem_drift.chat(filler)

print(f"  Completed {len(FILLER_MESSAGES)} filler turns.")
print(f"  Total summary versions: {len(mem_drift.summary_history)}")
print(f"  Total LLM summarization calls: {mem_drift._summary_calls}")

现在检查哪些事实在每个摘要版本中存活。我们在每份摘要里搜索关键词（比如 "elena vasquez" 或 "1992"）。对勾表示事实还在，叉号表示它已在压缩中丢失。


In [ ]:
# 检查哪些事实在每个摘要版本中存活
fact_labels = [f[0][:35] + "..." for f in FACTS]
fact_keywords = [f[1] for f in FACTS]

retention_matrix = []
for summary in mem_drift.summary_history:
    row = []
    for keyword in fact_keywords:
        present = keyword.lower() in summary.lower()
        row.append(1 if present else 0)
    retention_matrix.append(row)

# 打印保留率表格
print("Fact Retention Across Summary Versions")
print("=" * 70)
header = f"{'Version':<10}" + "".join(f"{label:<12}" for label in ["Elena V.", "Born 1992", "3 cats", "$145K", "Strawberry"])
print(header)
print("-" * 70)
for i, row in enumerate(retention_matrix):
    cells_str = "".join(f"{'  ✓':<12}" if v else f"{'  ✗':<12}" for v in row)
    print(f"v{i+1:<9}{cells_str}")

total_facts = len(FACTS)
final_retained = sum(retention_matrix[-1]) if retention_matrix else 0
print(f"\nFinal retention: {final_retained}/{total_facts} facts survived to the last summary version.")

把事实保留情况按摘要版本可视化。左侧热力图显示每条事实在哪些版本存活，右侧柱状图显示保留事实的总数。观察这个总数如何随摘要循环的累积而下降。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

versions = list(range(1, len(retention_matrix) + 1))
fact_names = ["Elena V.", "Born 1992", "3 Cats", "$145K Salary", "Strawberry Allergy"]

# 左图：事实保留热力图
retention_arr = np.array(retention_matrix)
im = ax1.imshow(retention_arr.T, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
ax1.set_xticks(range(len(versions)))
ax1.set_xticklabels([f"v{v}" for v in versions], fontsize=8)
ax1.set_yticks(range(len(fact_names)))
ax1.set_yticklabels(fact_names)
ax1.set_xlabel("Summary Version")
ax1.set_title("Fact Retention Heatmap\n(Green = present, Red = lost)")

# 添加文字标注
for i in range(len(versions)):
    for j in range(len(fact_names)):
        symbol = "✓" if retention_arr[i, j] else "✗"
        ax1.text(i, j, symbol, ha="center", va="center",
                 color="white" if retention_arr[i, j] == 0 else "black", fontsize=10)

# 右图：每个版本保留的事实总数
totals = [sum(row) for row in retention_matrix]
colors = ["#22c55e" if t >= 4 else "#f59e0b" if t >= 2 else "#ef4444" for t in totals]
ax2.bar(range(len(versions)), totals, color=colors, alpha=0.85)
ax2.set_xticks(range(len(versions)))
ax2.set_xticklabels([f"v{v}" for v in versions])
ax2.set_ylabel("Facts Retained")
ax2.set_ylim(0, len(FACTS) + 0.5)
ax2.set_xlabel("Summary Version")
ax2.set_title("Total Facts Retained per Summary Version")
ax2.axhline(y=len(FACTS), color="gray", linestyle="--", alpha=0.3, label=f"All {len(FACTS)} facts")
ax2.legend()

for i, t in enumerate(totals):
    ax2.text(i, t + 0.15, str(t), ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("summary_drift.png", dpi=150, bbox_inches="tight")
plt.show()

print("Summary drift is visible: facts degrade as summarization cycles accumulate.")

打印每个摘要版本的完整文本。按顺序读下来，你会看到措辞如何变化、细节如何脱落。这就是摘要漂移的实际过程。


In [ ]:
print("=== Full Summary Evolution ===\n")
for i, s in enumerate(mem_drift.summary_history):
    retained = sum(retention_matrix[i])
    print(f"--- Version {i+1} ({retained}/{len(FACTS)} facts retained) ---")
    print(s)
    print()

## token 成本：摘要记忆 vs. 完整缓冲区 vs. 滑动窗口

摘要记忆介于完整缓冲区（无限保留、成本二次增长）和滑动窗口（硬截断、成本恒定）之间。我们来比较这三种方案。


In [ ]:
import matplotlib.pyplot as plt

# 模拟 30 轮对话的 token 增长
# 假设：每条约 50 token，系统提示词约 30 token，摘要约 100 token
MSG_TOKENS = 50
SYS_TOKENS = 30
SUMMARY_TOKENS = 100
NUM_TURNS = 30
WINDOW_K = 10
BUFFER_SIZE = 6  # 每 3 轮摘要一次

buffer_cost, window_cost, summary_cost = [], [], []

for turn in range(1, NUM_TURNS + 1):
    n_msgs = turn * 2

    # 完整缓冲区：所有消息
    buffer_cost.append(SYS_TOKENS + n_msgs * MSG_TOKENS)

    # 滑动窗口：上限为 k
    window_cost.append(SYS_TOKENS + min(n_msgs, WINDOW_K) * MSG_TOKENS)

    # 摘要记忆：摘要 + 缓冲区（缓冲区周期性清空）
    buffer_msgs = (n_msgs % BUFFER_SIZE) or BUFFER_SIZE
    summary_cost.append(SYS_TOKENS + SUMMARY_TOKENS + buffer_msgs * MSG_TOKENS)

turns = list(range(1, NUM_TURNS + 1))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(turns, buffer_cost, "o-", color="#ef4444", label="Full Buffer", linewidth=2, markersize=4)
ax.plot(turns, window_cost, "s-", color="#22c55e", label=f"Sliding Window (k={WINDOW_K})", linewidth=2, markersize=4)
ax.plot(turns, summary_cost, "^-", color="#6366f1", label=f"Summary Memory (buf={BUFFER_SIZE})", linewidth=2, markersize=4)

ax.set_xlabel("Conversation Turn")
ax.set_ylabel("Input Tokens per API Call")
ax.set_title("Input Tokens per Turn: Three Memory Strategies")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("cost_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"At turn {NUM_TURNS}:")
print(f"  Full buffer:    {buffer_cost[-1]:,} tokens")
print(f"  Sliding window: {window_cost[-1]:,} tokens")
print(f"  Summary memory: {summary_cost[-1]:,} tokens")
print(f"\nNote: Summary memory also incurs extra cost for summarization calls (not shown).")

## 回忆测试：智能体还能用上摘要吗？

验证智能体能否真正用摘要里的事实回答问题。记住：原始消息它已经没有了。


In [ ]:
RECALL_QUESTIONS = [
    ("What is my full name?", "elena"),
    ("When was I born?", "1992"),
    ("What are my cats' names?", "mochi"),
    ("What is my salary?", "145"),
    ("What food am I allergic to?", "strawberr"),
]

print("=== Recall Test (after 20 total turns) ===\n")
results = []
for question, keyword in RECALL_QUESTIONS:
    answer = mem_drift.chat(question)
    recalled = keyword.lower() in answer.lower()
    status = "✓ RECALLED" if recalled else "✗ FORGOTTEN"
    results.append(recalled)
    print(f"  Q: {question}")
    print(f"  A: {answer[:120]}")
    print(f"  {status}")
    print()

score = sum(results)
print(f"Score: {score}/{len(RECALL_QUESTIONS)} facts recalled from summary.")

## 缓解摘要漂移

摘要漂移是这项技术的固有属性，但有几种策略可以减轻它的影响：

### 1. 更好的摘要提示词
明确写出要保留什么：
```
Preserve ALL of the following from the conversation:
- Proper nouns (names, places, organizations)
- Numbers (dates, amounts, quantities)
- Stated preferences and constraints
- Decisions made and their rationale
```

### 2. 结构化摘要
不用自由文本，改用结构化格式：
```
FACTS: [name: Elena Vasquez, born: 1992-03-15, ...]
DECISIONS: [chose Python over Java, ...]
OPEN QUESTIONS: [budget not yet decided, ...]
```
这样 LLM 更难不小心丢掉字段。

### 3. 更大的缓冲区
摘要循环越少，漂移越少。如果上下文窗口允许，用 `max_buffer_size=20` 而不是 `max_buffer_size=4`。

### 4. 摘要 + 缓冲区混合
在摘要之外，逐字保留最近 *k* 条消息。这是生产系统的做法，比如 LangChain 的 `ConversationSummaryBufferMemory`。我们在[技术 04](../04_summary_buffer_memory/)里介绍它。

### 5. 实体抽取
把关键实体抽取到一个独立的存储里，与摘要一起注入。即使摘要漂移了，实体存储仍保留结构化的事实。

### 6. 定期全量重新摘要
不要总是增量融合，偶尔从更大一段原始历史重新摘要。这能减少「摘要链」带来的误差累积。

## 讨论与权衡

### 优势
- **不限长度的对话**：与滑动窗口不同，摘要记忆能处理任意长的对话，不会丢掉全部较早的上下文。
- **token 成本有界**：摘要很紧凑。每轮成本大致恒定（摘要 + 缓冲区）。
- **优雅退化**：不是硬截断，而是信息逐渐压缩。重要主题往往比次要细节存活更久。
- **压缩灵活**：摘要提示词决定保留什么，你可以针对自己的领域调整。

### 局限
- **摘要漂移**：反复压缩带来累积的信息损失。具体数字、名字和细节最脆弱。
- **额外的 LLM 调用**：每轮摘要循环都消耗 token 并增加延迟（即响应到达前的等待）。缓冲区小到 4 时，每 2 轮就要多一次 LLM 调用。
- **记忆不确定**：同一对话跑两次，可能得到不同摘要，进而导致智能体行为不同。
- **难以调试**：智能体「忘了」某事时，你很难判断是摘要器丢掉了它，还是聊天模型忽略了它。
- **无法精确复述**：对已摘要的历史，智能体永远无法逐字引用。

### 何时使用摘要记忆

| 场景 | 建议 |
|----------|---------------|
| 长对话（50 轮以上） | 合适。控制成本的同时保住要点。 |
| 需要精确复述早期事实 | 不理想。改用缓冲区或检索增强的记忆。 |
| 成本敏感的应用 | 合适。比完整缓冲区便宜得多。 |
| 短对话（10 轮以内） | 开销不值得。用缓冲区或滑动窗口。 |
| 多会话智能体 | 合适。摘要在会话之间容易传递。 |

### 成本模型
对话 *n* 轮、缓冲区大小 *b* 时：
- **聊天调用：** *n* 次（与任何方案相同）
- **摘要调用：** floor(*n* / (*b*/2)) 次（每清空一次缓冲区一次）
- **额外总成本：** 与 *n/b* 成正比。只要 *b* 合理，这只占总成本的一小部分。


## 延伸阅读

- [LangChain ConversationSummaryMemory](https://python.langchain.com/docs/modules/memory/types/summary?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：生产级实现，提示词可定制
- [LangChain ConversationSummaryBufferMemory](https://python.langchain.com/docs/modules/memory/types/summary_buffer?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：摘要加 token 限量缓冲区的混合方案
- [Anthropic: Building Conversational AI](https://docs.anthropic.com/en/docs/build-with-claude/conversational-ai?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：官方多轮对话模式
- [Recursive Summarization with LLMs](https://arxiv.org/abs/2301.13848)：迭代式摘要质量的研究
- [Lilian Weng, "LLM Powered Autonomous Agents"](https://lilianweng.github.io/posts/2023-06-23-agent/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：综述，涵盖记忆架构

---

*← 上一篇：02 滑动窗口记忆 · 下一篇：[04 摘要缓冲记忆](../04_summary_buffer_memory/) →*

In [ ]:
# 清理演示过程中生成的临时文件
import os
for f in ["summary_drift.png", "cost_comparison.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 动手试一试

三个小挑战，帮你加深理解。每个大约需要 10-30 分钟。

### 挑战 1：为摘要做提示词工程
写三个不同的 `SUMMARIZER_PROMPT` 变体：一个优先保留事实，一个优先保留用户偏好，一个优先保留行动项。用每个变体跑同样的 10 轮对话，并排比较得到的摘要。

### 挑战 2：量化摘要漂移
把漂移实验扩展到 30 轮。每轮摘要后，用 `extract_topic_tags()` 捕捉主题。画出「原始轮次主题」存活数量随时间的变化，并计算一个主题在摘要中的半衰期。

### 挑战 3：两层摘要
维护两份摘要：一份详细的（最近 10 轮），一份高层次的（更早的全部）。当详细摘要超过阈值时，把它压缩进高层次摘要。这对应 15 记忆压缩（Memory Compaction）里的模式。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--03-summary-memory--summary-memory)
